In [70]:
!pip install pyspark

In [71]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [72]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("UK Online Retail Analytics")
    .master("local[*]")
    .getOrCreate()
)

In [73]:
spark

In [74]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [75]:
df = spark.read.csv(
    "/content/drive/MyDrive/Pyspark_Project/data/data.csv",
    header=True,
    inferSchema=True
)

In [76]:
from pyspark.sql.functions import col

for column in df.columns:
    print(column, ":", df.filter(col(column).isNull()).count())

InvoiceNo : 0
StockCode : 0
Description : 1454
Quantity : 0
InvoiceDate : 0
UnitPrice : 0
CustomerID : 135080
Country : 0


**REMOVING MISSING CUSTOMER ID(S)**

In [77]:
df = df.na.drop(subset=["CustomerID"])

In [78]:
#verification
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|          0|       0|          0|        0|         0|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+



**REMOVE DUPLICATES**

In [79]:
df.count()

406829

In [80]:
df= df.drop_duplicates()

In [81]:
df.count()

401604

**REMOVE CANCELLED ORDERS**

In [82]:
df.filter(col("InvoiceNo").startswith("C")).show()

+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|  C538062|    22168|ORGANISER WOOD AN...|      -2| 12/9/2010 13:40|      8.5|     17702|United Kingdom|
|  C538118|   47594A|CAROUSEL DESIGN W...|      -1| 12/9/2010 15:33|     1.95|     17700|United Kingdom|
|  C538691|    22645|CERAMIC HEART FAI...|      -2| 12/14/2010 9:56|     1.45|     14176|United Kingdom|
|  C539657|    21876|       POTTERING MUG|      -1|12/20/2010 17:24|     1.25|     16134|United Kingdom|
|  C540787|    22360|GLASS JAR ENGLISH...|      -4| 1/11/2011 11:45|     2.55|     13408|United Kingdom|
|  C541573|    22778|  GLASS CLOCHE SMALL|      -4| 1/19/2011 12:47|     3.39|     15311|United Kingdom|
|  C542138|    20857|BLUE ROSE PATCH P...|     -15| 1/2

In [83]:
df.filter(col("InvoiceNo").startswith("C")).count()

8872

In [84]:
df = df.filter(~col("InvoiceNo").startswith("C"))

In [85]:
df.count()

392732

**CHECK FOR INVALID QUANITTY AND UNIT PRICE**

In [86]:
df.filter(col("quantity")<0).count()

0

In [87]:
df.filter(col("UnitPrice")<0).count()

0

**CONVERT INVOICE-DATE**

In [88]:
df = df.withColumn(
    "InvoiceDate",
    to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")
)

In [89]:
df.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536381|    21731|RED TOADSTOOL LED...|       2|2010-12-01 09:41:00|     1.65|     15311|United Kingdom|
|   536390|    22174|          PHOTO CUBE|      48|2010-12-01 10:19:00|     1.48|     17511|United Kingdom|
|   536396|    82483|WOOD 2 DRAWER CAB...|       2|2010-12-01 10:51:00|     4.95|     17850|United Kingdom|
|   536408|    22914|BLUE COAT RACK PA...|       3|2010-12-01 11:41:00|     4.95|     14307|United Kingdom|
|   536412|    22382|LUNCH BAG SPACEBO...|       3|2010-12-01 11:49:00|     1.65|     17920|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows


**REMOVING SPACES**

In [90]:
df = df.withColumn("Description", trim(col("Description")))

In [91]:
df.show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536381|    21731|RED TOADSTOOL LED...|       2|2010-12-01 09:41:00|     1.65|     15311|United Kingdom|
|   536390|    22174|          PHOTO CUBE|      48|2010-12-01 10:19:00|     1.48|     17511|United Kingdom|
|   536396|    82483|WOOD 2 DRAWER CAB...|       2|2010-12-01 10:51:00|     4.95|     17850|United Kingdom|
|   536408|    22914|BLUE COAT RACK PA...|       3|2010-12-01 11:41:00|     4.95|     14307|United Kingdom|
|   536412|    22382|LUNCH BAG SPACEBO...|       3|2010-12-01 11:49:00|     1.65|     17920|United Kingdom|
|   536416|    22767|TRIPLE PHOTO FRAM...|       4|2010-12-01 11:58:00|     9.95|     13255|United Kingdom|
|   536514|    22807|SET OF 

In [92]:
df = df.withColumn(
    "TotalAmount",
    round(col("Quantity") * col("UnitPrice"),2)
)

In [93]:
df.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|TotalAmount|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+
|   536381|    21731|RED TOADSTOOL LED...|       2|2010-12-01 09:41:00|     1.65|     15311|United Kingdom|        3.3|
|   536390|    22174|          PHOTO CUBE|      48|2010-12-01 10:19:00|     1.48|     17511|United Kingdom|      71.04|
|   536396|    82483|WOOD 2 DRAWER CAB...|       2|2010-12-01 10:51:00|     4.95|     17850|United Kingdom|        9.9|
|   536408|    22914|BLUE COAT RACK PA...|       3|2010-12-01 11:41:00|     4.95|     14307|United Kingdom|      14.85|
|   536412|    22382|LUNCH BAG SPACEBO...|       3|2010-12-01 11:49:00|     1.65|     17920|United Kingdom|       4.95|
+---------+---------+-------------------

**FEATURE ENGINEERING**

In [94]:
df = df.withColumn("Year",year("InvoiceDate"))
df = df.withColumn("Month",month("InvoiceDate"))
df = df.withColumn("MonthName",monthname("InvoiceDate"))
df = df.withColumn("Day",day("InvoiceDate"))
df = df.withColumn("Weekday",weekday("InvoiceDate"))

In [95]:
df.show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+----+-----+---------+---+-------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|TotalAmount|Year|Month|MonthName|Day|Weekday|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+----+-----+---------+---+-------+
|   536381|    21731|RED TOADSTOOL LED...|       2|2010-12-01 09:41:00|     1.65|     15311|United Kingdom|        3.3|2010|   12|      Dec|  1|      2|
|   536390|    22174|          PHOTO CUBE|      48|2010-12-01 10:19:00|     1.48|     17511|United Kingdom|      71.04|2010|   12|      Dec|  1|      2|
|   536396|    82483|WOOD 2 DRAWER CAB...|       2|2010-12-01 10:51:00|     4.95|     17850|United Kingdom|        9.9|2010|   12|      Dec|  1|      2|
|   536408|    22914|BLUE COAT RACK PA...|       3|2010-12-01 11:41:00|     4.95| 

In [96]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- TotalAmount: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- MonthName: string (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Weekday: integer (nullable = true)



In [97]:
df.count()

392732

**SAVING CLEANED DATASET**

In [99]:
df.write.mode("overwrite").parquet("/content/drive/MyDrive/Pyspark_Project/output/cleaned_data")